# PyTorch Transformer-Lite Representation Learning

This notebook learns compact representations of bounded hourly weather windows using CPU-friendly transformer-lite autoencoders. The benchmark compares several small encoder settings side by side and inspects the resulting embedding structure.

The workflow is:

- build bounded 24-hour windows from the canonical hourly table
- train transformer-lite autoencoders with reconstruction loss
- compare models by reconstruction error and embedding dispersion
- inspect PCA projections and recent high-error windows

In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device("cpu")
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print({"device": str(device), "torch_version": torch.__version__})

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {
    name: np.asarray(values, dtype=float) for name, values in table.columns.items()
}
timestamps = np.asarray(table.timestamps)

feature_names = [
    "misc__temperature_2m",
    "misc__relative_humidity_2m",
    "misc__pressure_msl",
    "misc__precipitation",
    "wind__wind_speed_10m",
    "wind__wind_gusts_10m",
    "cloud_cover__cloud_cover",
    "solar_radiation_vars__shortwave_radiation",
    "european_air_quality_index__european_aqi",
    "additional_vars_and_options__is_day",
]
missing = [name for name in feature_names if name not in column_arrays]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

feature_matrix = np.column_stack(
    [column_arrays[name] for name in feature_names]
).astype(np.float32)
window_size = 24
max_windows = 14000
windows = []
window_times = []
for end_index in range(window_size, len(timestamps) + 1):
    window = feature_matrix[end_index - window_size : end_index]
    if np.isfinite(window).all():
        windows.append(window)
        window_times.append(timestamps[end_index - 1])
X = np.asarray(windows, dtype=np.float32)
time_index = np.asarray(window_times)
if len(X) > max_windows:
    X = X[-max_windows:]
    time_index = time_index[-max_windows:]
print(f"Windows: {len(X):,}")
print(f"Window shape: {X.shape[1:]}")
print(f"Subset start: {time_index[0]}")
print(f"Subset end:   {time_index[-1]}")

In [ ]:
def chronological_split(
    values: np.ndarray,
    time_values: np.ndarray,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.15,
):
    total = len(values)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        "X_train": values[:train_end],
        "t_train": time_values[:train_end],
        "X_valid": values[train_end:valid_end],
        "t_valid": time_values[train_end:valid_end],
        "X_test": values[valid_end:],
        "t_test": time_values[valid_end:],
    }


split = chronological_split(X, time_index)
train_flat = split["X_train"].reshape(-1, split["X_train"].shape[-1])
x_mean = train_flat.mean(axis=0)
x_std = train_flat.std(axis=0) + 1e-6


def normalize_features(values: np.ndarray) -> np.ndarray:
    return ((values - x_mean) / x_std).astype(np.float32)


def to_tensor_dataset(values: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.from_numpy(normalize_features(values)))


train_loader = DataLoader(
    to_tensor_dataset(split["X_train"]), batch_size=64, shuffle=True, num_workers=0
)
valid_loader = DataLoader(
    to_tensor_dataset(split["X_valid"]), batch_size=128, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    to_tensor_dataset(split["X_test"]), batch_size=128, shuffle=False, num_workers=0
)
for name in ["train", "valid", "test"]:
    print(name, split[f"X_{name}"].shape, split[f"t_{name}"][0], split[f"t_{name}"][-1])

In [ ]:
class TransformerLiteAutoencoder(nn.Module):
    def __init__(
        self,
        input_size: int,
        seq_len: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        ff_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.input_projection = nn.Linear(input_size, d_model)
        self.position_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, input_size)

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        encoded = (
            self.input_projection(inputs) + self.position_embedding[:, : inputs.size(1)]
        )
        encoded = self.norm(self.encoder(encoded))
        reconstruction = self.output_projection(encoded)
        embedding = encoded.mean(dim=1)
        return reconstruction, embedding


def reconstruction_errors(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    errors = []
    with torch.no_grad():
        for (features,) in loader:
            reconstruction, _ = model(features)
            batch_errors = torch.mean((reconstruction - features) ** 2, dim=(1, 2))
            errors.append(batch_errors.detach().numpy())
    return np.concatenate(errors)


def collect_embeddings(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    outputs = []
    with torch.no_grad():
        for (features,) in loader:
            _, embedding = model(features)
            outputs.append(embedding.detach().numpy())
    return np.concatenate(outputs)


def train_model(
    model: nn.Module, train_data: DataLoader, valid_data: DataLoader, epochs: int = 8
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    history = []
    best_valid_loss = float("inf")
    best_state = None
    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for (features,) in train_data:
            optimizer.zero_grad()
            reconstruction, _ = model(features)
            loss = loss_fn(reconstruction, features)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))
        valid_loss = float(np.mean(reconstruction_errors(model, valid_data)))
        history.append(
            {"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss}
        )
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {
                name: tensor.detach().clone()
                for name, tensor in model.state_dict().items()
            }
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

In [ ]:
benchmark_specs = {
    "transformer_tiny": {
        "d_model": 32,
        "nhead": 4,
        "num_layers": 1,
        "ff_dim": 64,
        "dropout": 0.05,
    },
    "transformer_small": {
        "d_model": 48,
        "nhead": 4,
        "num_layers": 2,
        "ff_dim": 96,
        "dropout": 0.05,
    },
    "transformer_wide_ff": {
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "ff_dim": 128,
        "dropout": 0.1,
    },
}
benchmark_rows = []
histories = {}
embedding_results = {}
for model_name, spec in benchmark_specs.items():
    model = TransformerLiteAutoencoder(
        input_size=len(feature_names),
        seq_len=window_size,
        d_model=spec["d_model"],
        nhead=spec["nhead"],
        num_layers=spec["num_layers"],
        ff_dim=spec["ff_dim"],
        dropout=spec["dropout"],
    )
    history = train_model(model, train_loader, valid_loader)
    valid_errors = reconstruction_errors(model, valid_loader)
    test_errors = reconstruction_errors(model, test_loader)
    embeddings = collect_embeddings(model, test_loader)
    reduced = PCA(n_components=2, random_state=42).fit_transform(embeddings)
    clusters = KMeans(n_clusters=4, n_init=20, random_state=42).fit_predict(embeddings)
    benchmark_rows.append(
        {
            "model": model_name,
            "valid_error_mean": float(np.mean(valid_errors)),
            "test_error_mean": float(np.mean(test_errors)),
            "embedding_std_mean": float(np.mean(embeddings.std(axis=0))),
        }
    )
    histories[model_name] = history
    embedding_results[model_name] = {
        "test_errors": test_errors,
        "reduced": reduced,
        "clusters": clusters,
    }
for row in sorted(benchmark_rows, key=lambda item: item["test_error_mean"]):
    print(row)

In [ ]:
best_model_name = min(benchmark_rows, key=lambda item: item["test_error_mean"])["model"]
best_result = embedding_results[best_model_name]
fig, axes = plt.subplots(
    1, len(histories), figsize=(5 * len(histories), 4), sharey=True
)
axes = np.atleast_1d(axes)
for ax, (model_name, history) in zip(axes, histories.items()):
    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]
    valid_loss = [row["valid_loss"] for row in history]
    ax.plot(epochs, train_loss, label="train")
    ax.plot(epochs, valid_loss, label="valid")
    ax.set_title(model_name)
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("Reconstruction MSE")
plt.tight_layout()
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    best_result["reduced"][:, 0],
    best_result["reduced"][:, 1],
    c=best_result["clusters"],
    cmap="tab10",
    s=18,
    alpha=0.75,
)
ax.set_title(f"PCA view of embeddings for {best_model_name}")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.colorbar(scatter, ax=ax, label="KMeans cluster")
plt.tight_layout()
top_order = np.argsort(best_result["test_errors"])[-10:][::-1]
fig, ax = plt.subplots()
ax.plot(
    split["t_test"],
    best_result["test_errors"],
    label="reconstruction error",
    linewidth=1.4,
)
ax.scatter(
    split["t_test"][top_order],
    best_result["test_errors"][top_order],
    color="crimson",
    label="top high-error windows",
    zorder=3,
)
ax.set_title(f"Recent high-error windows for {best_model_name}")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Reconstruction error")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
print("Top high-error windows for best model:")
for index in top_order:
    print(split["t_test"][index], float(best_result["test_errors"][index]))